In [ ]:
# Cell 1: Import libraries and load datasets
from datasets import load_dataset, concatenate_datasets

print("Loading CultureBank...")
ds_dict = load_dataset("SALT-NLP/CultureBank")
ds_cb = concatenate_datasets([ds_dict['tiktok'], ds_dict['reddit']])

print("Loading MNLI (The new generic dataset)...")
ds_mnli = load_dataset("multi_nli", split="train")

print(f"CultureBank initial size: {len(ds_cb)}")
print(f"MNLI initial size: {len(ds_mnli)}")

Loading CultureBank...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/186 [00:00<?, ?B/s]

tiktok/culturebank_tiktok.csv:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

reddit/culturebank_reddit.csv:   0%|          | 0.00/20.7M [00:00<?, ?B/s]

...

data/validation_mismatched-00000-of-0000(…):   0%|          | 0.00/5.10M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/392702 [00:00<?, ? examples/s]

Generating validation_matched split:   0%|          | 0/9815 [00:00<?, ? examples/s]

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

CultureBank initial size: 22990
MNLI initial size: 392702


In [ ]:
# Cell 2: Format columns for consistency
print("Formatting datasets...")

# Format Class 1 (Norms) using 'actor_behaviour'
ds_cb = ds_cb.rename_column("actor_behaviour", "text")
ds_cb = ds_cb.select_columns(["text"])
ds_cb = ds_cb.add_column("label", [1] * len(ds_cb))

# Format Class 0 (Generic) using 'premise'
ds_mnli = ds_mnli.rename_column("premise", "text")
ds_mnli = ds_mnli.select_columns(["text"])
ds_mnli = ds_mnli.add_column("label", [0] * len(ds_mnli))

print("Columns successfully mapped and standardized!")

Formatting datasets...


ValueError: Original column name actor_behaviour not in the dataset. Current columns in the dataset: ['cultural group', 'context', 'goal', 'relation', 'actor', 'actor_behavior', 'recipient', 'recipient_behavior', 'other_descriptions', 'topic', 'agreement', 'num_support_bin', 'time_range', 'eval_whole_desc', 'eval_scenario', 'eval_persona', 'eval_question']

In [ ]:
# Cell 3: Filter by length, deduplicate, and balance classes
import pandas as pd
from datasets import Dataset

print("Filtering MNLI to match CultureBank's sentence length...")
mnli_df = ds_mnli.to_pandas()

# Create a word count column
mnli_df['word_count'] = mnli_df['text'].apply(lambda x: len(str(x).split()))

# Filter out very short sentences (adjusted to > 8 for actor_behaviour)
long_mnli_df = mnli_df[mnli_df['word_count'] > 8]

# Deduplicate
long_mnli_df = long_mnli_df.drop_duplicates(subset=["text"])

# Convert back to Hugging Face
ds_mnli_filtered = Dataset.from_pandas(long_mnli_df, preserve_index=False)
ds_mnli_filtered = ds_mnli_filtered.remove_columns(["word_count"])

# Downsample to match CultureBank exactly
ds_generic = ds_mnli_filtered.shuffle(seed=42).select(range(len(ds_cb)))

print(f"Class 1 (Norms) size: {len(ds_cb)}")
print(f"Class 0 (Generic) size: {len(ds_generic)}")

In [ ]:
# Cell 3.5: Diagnostic - Lexical and Punctuation Bias
print("--- Running Bias Diagnostics ---\n")

cb_df = ds_cb.to_pandas()
generic_df = ds_generic.to_pandas()

def has_period(text):
    return str(text).strip().endswith('.')

cb_periods = cb_df['text'].apply(has_period).mean() * 100
generic_periods = generic_df['text'].apply(has_period).mean() * 100

print("PUNCTUATION BIAS:")
print(f"CultureBank ending in a period: {cb_periods:.2f}%")
print(f"MNLI (Generic) ending in a period: {generic_periods:.2f}%\n")

cheat_words = ['customary', 'common', 'norm', 'culture', 'tradition', 'practice', 'expected']
def check_keywords(text):
    text_lower = str(text).lower()
    return any(word in text_lower for word in cheat_words)

cb_lexical_bias = cb_df['text'].apply(check_keywords).mean() * 100
print("LEXICAL BIAS:")
print(f"CultureBank containing obvious keywords: {cb_lexical_bias:.2f}%")

In [ ]:
# Cell 3.6: Universal Sanitizer (Removing Lexical & Full-Stop Bias)
import re
import string

def sanitize_and_debias(example):
    text = str(example["text"])

    # 1. Remove lexical cheat words
    words_to_remove = r'\b(customary|common practice|norm|culture|tradition|expected)\b'
    text = re.sub(words_to_remove, '', text, flags=re.IGNORECASE)

    # 2. Remove location giveaways (e.g., "In America,")
    location_pattern = r'^In\s+[A-Z][a-z]+\s*(culture|society)?\s*,'
    text = re.sub(location_pattern, '', text)

    # 3. Strip periods and trailing punctuation to fix full-stop bias
    text = text.rstrip(string.punctuation).strip()

    # 4. Clean up spacing
    text = re.sub(r'\s+', ' ', text)

    example["text"] = text
    return example

print("Sanitizing BOTH datasets to neutralize bias...")
ds_cb = ds_cb.map(sanitize_and_debias)
ds_generic = ds_generic.map(sanitize_and_debias)
print("Sanitization complete!")

In [ ]:
# Cell 4: Merge datasets and clean empty rows
ds_combined = concatenate_datasets([ds_cb, ds_generic])
ds_combined = ds_combined.shuffle(seed=42)
ds_combined = ds_combined.filter(lambda x: x["text"] is not None and len(x["text"].strip()) > 0)

print(f"Total unified dataset size: {len(ds_combined)} rows\n")

In [ ]:
# Cell 5: Split into Train (80%), Val (10%), Test (10%)
from datasets import ClassLabel

ds_combined = ds_combined.cast_column("label", ClassLabel(num_classes=2, names=["generic", "norm"]))

split_1 = ds_combined.train_test_split(test_size=0.20, seed=42, stratify_by_column="label")
train_ds = split_1["train"]
temp_ds = split_1["test"]

split_2 = temp_ds.train_test_split(test_size=0.50, seed=42, stratify_by_column="label")
val_ds = split_2["train"]
test_ds = split_2["test"]

print(f"Train size: {len(train_ds)} | Val size: {len(val_ds)} | Test size: {len(test_ds)}")

In [ ]:
# Cell 6: Tokenize text
from transformers import AutoTokenizer

print("Initializing tokenizers...")
tokenizer_bert = AutoTokenizer.from_pretrained("bert-base-uncased")
tokenizer_roberta = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_bert(examples):
    return tokenizer_bert(examples["text"], padding="max_length", truncation=True, max_length=64)

def tokenize_roberta(examples):
    return tokenizer_roberta(examples["text"], padding="max_length", truncation=True, max_length=64)

print("Applying tokenizers...")
train_tok_bert = train_ds.map(tokenize_bert, batched=True)
val_tok_bert = val_ds.map(tokenize_bert, batched=True)

train_tok_roberta = train_ds.map(tokenize_roberta, batched=True)
val_tok_roberta = val_ds.map(tokenize_roberta, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_tok_bert.set_format("torch", columns=cols); val_tok_bert.set_format("torch", columns=cols)
train_tok_roberta.set_format("torch", columns=cols); val_tok_roberta.set_format("torch", columns=cols)
print("Tokenization complete!")

In [ ]:
# Cell 7: Set up PyTorch DataLoaders
from torch.utils.data import DataLoader

BATCH_SIZE = 16

loader_train_bert = DataLoader(train_tok_bert, batch_size=BATCH_SIZE, shuffle=True)
loader_val_bert = DataLoader(val_tok_bert, batch_size=BATCH_SIZE)

loader_train_roberta = DataLoader(train_tok_roberta, batch_size=BATCH_SIZE, shuffle=True)
loader_val_roberta = DataLoader(val_tok_roberta, batch_size=BATCH_SIZE)

print(f"DataLoaders ready! {len(loader_train_bert)} batches per epoch.")

In [ ]:
# Cell 8: Initialize Transformer Models
import torch
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware utilized: {device}\n")

NUM_LABELS = 2

print("Downloading Model 1: BERT Base...")
model_bert = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=NUM_LABELS).to(device)

print("Downloading Model 2: RoBERTa Base...")
model_roberta = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=NUM_LABELS).to(device)

print("Downloading Model 3: DistilBERT...")
model_distilbert = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=NUM_LABELS).to(device)

In [ ]:
# Cell 10: The Bake-off and Testing Function
print("Initiating the Bake-off...")

# Run just 1 epoch first to make sure it doesn't mode-collapse!
model_bert = train_model(model_bert, loader_train_bert, loader_val_bert, "BERT", epochs=3)
model_roberta = train_model(model_roberta, loader_train_roberta, loader_val_roberta, "RoBERTa", epochs=3)
model_distilbert = train_model(model_distilbert, loader_train_bert, loader_val_bert, "DistilBERT", epochs=3)

# --- INFERENCE FUNCTION ---
def predict_text(text, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: "Generic", 1: "Norm"}
    return label_map[prediction]

In [ ]:
# Cell 9: Define the Custom Training Loop
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm.auto import tqdm

def train_model(model, train_loader, val_loader, model_name, epochs=3):
    print(f"\n{'='*40}\n--- Starting Training for {model_name} ---\n{'='*40}")

    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

    for epoch in range(epochs):
        print(f"\nEpoch {epoch + 1}/{epochs}")

        # --- TRAINING PHASE ---
        model.train()
        total_train_loss = 0
        progress_bar = tqdm(train_loader, desc="Training")

        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).long()

            model.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_train_loss += loss.item()

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()
            progress_bar.set_postfix({'loss': loss.item()})

        avg_train_loss = total_train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        model.eval()
        total_val_loss = 0
        correct_predictions = 0
        total_predictions = 0

        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device).long()

                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()

                predictions = torch.argmax(outputs.logits, dim=-1)
                correct_predictions += (predictions == labels).sum().item()
                total_predictions += labels.size(0)

        avg_val_loss = total_val_loss / len(val_loader)
        val_accuracy = correct_predictions / total_predictions

        print(f"-> Train Loss: {avg_train_loss:.4f}")
        print(f"-> Val Loss:   {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.2%}")

    print(f"--- Training Complete for {model_name} ---")
    return model

# --- INFERENCE FUNCTION (Defined here for later use) ---
def predict_text(text, model, tokenizer):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=64)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    prediction = torch.argmax(outputs.logits, dim=-1).item()
    label_map = {0: "Generic", 1: "Norm"}
    return label_map[prediction]

In [ ]:
# Cell 10: Run the Multi-Model Fine Tuning Bake-off
print("Initiating the Bake-off...")

model_bert = train_model(model_bert, loader_train_bert, loader_val_bert, "BERT", epochs=3)
model_roberta = train_model(model_roberta, loader_train_roberta, loader_val_roberta, "RoBERTa", epochs=3)
model_distilbert = train_model(model_distilbert, loader_train_bert, loader_val_bert, "DistilBERT", epochs=3)

In [ ]:
# Cell 11: Final Evaluation on the Unseen Test Set
from sklearn.metrics import classification_report

print("Tokenizing the Test Set...")
test_tok_bert = test_ds.map(tokenize_bert, batched=True)
test_tok_roberta = test_ds.map(tokenize_roberta, batched=True)

cols = ["input_ids", "attention_mask", "label"]
test_tok_bert.set_format("torch", columns=cols)
test_tok_roberta.set_format("torch", columns=cols)

loader_test_bert = DataLoader(test_tok_bert, batch_size=16)
loader_test_roberta = DataLoader(test_tok_roberta, batch_size=16)

def evaluate_final_model(model, test_loader, model_name):
    model.eval()
    all_preds = []
    all_labels = []

    print(f"\nEvaluating {model_name}...")
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device).long()

            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"--- {model_name} Final Test Results ---")
    print(classification_report(all_labels, all_preds, target_names=["Generic (0)", "Norm (1)"]))

evaluate_final_model(model_bert, loader_test_bert, "BERT")
evaluate_final_model(model_roberta, loader_test_roberta, "RoBERTa")
evaluate_final_model(model_distilbert, loader_test_bert, "DistilBERT")

In [ ]:
# Cell 12: Manual Adversarial Testing (Fixed Pandas Styler)
import pandas as pd

test_sentences = [
    "Tipping 20 percent at restaurants is considered polite.",
    "The mitochondria is the powerhouse of the cell.",
    "Employees are expected to arrive by 9:00 AM.",
    "The Eiffel Tower is located in Paris, France.",
    "Water must reach 100 degrees Celsius to boil.",
    "Guests bring gifts to weddings.",
    "Men take off their hats when entering a church.",
    "A triangle has exactly three sides and three angles.",
    "You should brush your teeth twice a day.",
    "Planets orbit the sun due to gravitational pull."
]

expected_labels = ["Norm", "Generic", "Norm", "Generic", "Generic", "Norm", "Norm", "Generic", "Norm", "Generic"]

print("Running Manual Adversarial Test...\n")

results = []
for i, text in enumerate(test_sentences):
    # Sanitize inputs exactly as we did the training data to ensure consistency
    text = text.rstrip('.').strip()

    pred_bert = predict_text(text, model_bert, tokenizer_bert)
    pred_roberta = predict_text(text, model_roberta, tokenizer_roberta)
    pred_distilbert = predict_text(text, model_distilbert, tokenizer_bert)

    results.append({
        "Sentence": text,
        "Expected": expected_labels[i],
        "BERT": pred_bert,
        "RoBERTa": pred_roberta,
        "DistilBERT": pred_distilbert
    })

df_results = pd.DataFrame(results)

def highlight_errors(row):
    colors = ['' for _ in row]
    columns_to_color = ["BERT", "RoBERTa", "DistilBERT"]

    for col in columns_to_color:
        col_idx = row.index.get_loc(col)
        if row[col] != row["Expected"]:
            colors[col_idx] = 'background-color: lightcoral'
        else:
            colors[col_idx] = 'background-color: lightgreen'
    return colors

styled_df = df_results.style.apply(highlight_errors, axis=1)
display(styled_df)

In [ ]:
# Cell 13: Save Models to Google Drive
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/NormClassifier_Models'
os.makedirs(save_dir, exist_ok=True)

print(f"\nSaving models to: {save_dir}")

print("Saving BERT...")
model_bert.save_pretrained(f"{save_dir}/BERT_Model")
tokenizer_bert.save_pretrained(f"{save_dir}/BERT_Model")

print("Saving RoBERTa...")
model_roberta.save_pretrained(f"{save_dir}/RoBERTa_Model")
tokenizer_roberta.save_pretrained(f"{save_dir}/RoBERTa_Model")

print("Saving DistilBERT...")
model_distilbert.save_pretrained(f"{save_dir}/DistilBERT_Model")
tokenizer_bert.save_pretrained(f"{save_dir}/DistilBERT_Model")

print("\nBackup Complete!")